# Rede circular de atribuição de emissões — MIP brasileira de 67 setores

Figura independente da apresentação. Cada setor tem dois papéis: **origem emissora**
e **destino da atribuição por demanda final**. Reunimos os dois papéis num único nó:
área = emissões da produção; cor = emissões do consumo; ligação i→j = Fij.
Não é uma rede de transações diretas Z nem de fluxos físicos de CO₂.

O nó neutro **Demanda final** reúne as 67 contas u_j. Suas ligações são um fechamento
visual das colunas de F, não uma segunda atribuição a adicionar às emissões totais.
Todos os 67 setores e todas as parcelas positivas de F, incluindo a diagonal, são mantidos.

Execute as células em ordem a partir da raiz do projeto. As fontes e escolhas são as
mesmas de `sankeys_mip20_67.ipynb`; nenhum resultado histórico ou slide é substituído.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch
from matplotlib.path import Path as MplPath
from matplotlib.colors import Normalize
from matplotlib.ticker import FuncFormatter
from redes import dados, modelo, mip

pasta = dados.RAIZ_PROJETO / 'outputs/rede_circular_atribuicao_mip67'
pasta.mkdir(parents=True, exist_ok=True)
tabelas = dados.carregar_matriz_67('mip_ibge_2015_67')
x = modelo.calcular_producao_bruta_67(tabelas)
Z = modelo.calcular_matriz_transacoes_intersetoriais_67(tabelas)
y = mip.carregar_demanda_final_nacional_67(tabelas)
L = mip.carregar_inversa_leontief_ibge_67(tabelas)
codigos = x.index.tolist()
nomes = tabelas['14'].iloc[5:72, 1].astype(str).str.strip().tolist()
phi_2011 = dados.carregar_coeficientes_co2('sanguinet_azzoni_2011')[2011]
phi_2018 = dados.carregar_coeficientes_co2('sanguinet_azzoni_2018')[2018]

# S1...S67 identificam a ordem da tabela ambiental, não os códigos IBGE.
# Verificamos essa ordem antes do mapeamento explícito para as atividades IBGE.
ordem_ambiental = [f'S{i}' for i in range(1, 68)]
assert phi_2011.index.tolist() == ordem_ambiental
assert phi_2018.index.tolist() == ordem_ambiental
assert all(obj.index.equals(x.index) for obj in [Z, y, L])
assert Z.columns.equals(x.index) and L.columns.equals(x.index)
assert len(set(codigos)) == len(nomes) == 67
phi = pd.Series((phi_2011 + 4/7 * (phi_2018 - phi_2011)).to_numpy(),
                index=x.index, name='phi_gg_co2_por_r_milhao')

SHA-256 verificado: Matriz_de_Insumo_Produto_2015_Nivel_67.xls.original
SHA-256 verificado: coeficientes_co2_sanguinet_azzoni_2011.csv
SHA-256 verificado: coeficientes_co2_sanguinet_azzoni_2018.csv


## Contas e fronteira

\(\Phi=\widehat\phi\), \(F=\Phi L\widehat y\), \(e=\Phi x=F\mathbf1\)
e \(u=F^\top\mathbf1\). Fij está em **Gg de CO₂**; x e y em **R$ milhões**.

Produção nacional de 2015, com exportações na demanda final. Emissões importadas
excluídas. Portanto u é a atribuição doméstica à demanda final por produtos nacionais,
não a pegada integral do consumo dos residentes.

As emissões são estimativas: interpolação linear dos coeficientes de Sanguinet e
Azzoni (2011/2018) para 2015, fator de compatibilidade monetária 1, sem deflação
adicional. Não constituem um inventário ambiental oficial observado em 2015.

In [2]:
e = (phi * x).rename('emissoes_producao_gg_co2')
F = L.mul(phi, axis=0).mul(y, axis=1)
F.index.name = 'emissor_ibge'
F.columns.name = 'destino_demanda_final_ibge'
u = F.sum(axis=0).rename('emissoes_consumo_gg_co2')
np.testing.assert_allclose(Z.sum(axis=1) + y, x, rtol=1e-12, atol=1e-7)
np.testing.assert_allclose(L @ y, x, rtol=1e-12, atol=1e-7)
A = Z.to_numpy() / x.to_numpy()[None, :]
np.testing.assert_allclose(L, np.linalg.inv(np.eye(67)-A), rtol=1e-10, atol=1e-12)
np.testing.assert_allclose(F.sum(axis=1), e, rtol=1e-12, atol=1e-7)
np.testing.assert_allclose(F.sum(axis=0), u, rtol=1e-12, atol=1e-7)
np.testing.assert_allclose(u.sum(), e.sum(), rtol=1e-12, atol=1e-7)
assert all(np.isfinite(obj.to_numpy()).all() for obj in [x, y, phi, e, F, u])
assert (F.to_numpy() >= 0).all() and (e > 0).all() and (u > 0).all()
assert int((F.to_numpy() > 0).sum()) == 4357
np.testing.assert_allclose(e.sum(), 683696.0414285716, rtol=1e-12)
setores = pd.DataFrame({
    'codigo_ibge': codigos, 'nome': nomes, 'codigo_ambiental': ordem_ambiental,
    'x_r_milhao': x.to_numpy(), 'y_r_milhao': y.to_numpy(),
    'phi_gg_co2_por_r_milhao': phi.to_numpy(),
    'e_producao_gg_co2': e.to_numpy(), 'u_consumo_gg_co2': u.to_numpy(),
    'ordem_circular': e.rank(method='first', ascending=False).astype(int).to_numpy(),
})
print(f'67 setores; {(F.to_numpy()>0).sum()} parcelas positivas; total = {e.sum():.8f} Gg CO2')

67 setores; 4357 parcelas positivas; total = 683696.04142857 Gg CO2


## Desenho

Ordem decrescente de emissões pela produção (área), sentido horário, começando no topo. Área = k·e, com um único k
ajustado geometricamente para evitar sobreposição dos círculos. Cor linear em u,
paleta viridis invertida. Espessura = k_F·massa, com o **mesmo fator** para Fij e u_j;
sem transformação logarítmica ou supressão. Para evidenciar a densidade, as parcelas de F
com largura inferior a 0,45 pt recebem piso visual de 0,45 pt e cinza claro. Os pesos
originais permanecem intactos; essas linhas indicam presença, não sua magnitude pela espessura.

As setas cinza são Fij e as azuis são os fechamentos u_j. Sua transparência é igual.
Autoligações conservam Fii, inclusive os retornos da cadeia presentes em Lii.
Os papéis emissor/destino coexistem em cada nó: não interpretar caminhos sucessivos
neste desenho como uma nova multiplicação de F ou como etapas físicas de produção.

In [3]:
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 12,
                     'svg.fonttype': 'none', 'pdf.fonttype': 42})
fig = plt.figure(figsize=(24, 18), facecolor='white')
ax = fig.add_axes([.035, .19, .92, .67])
ax.set_aspect('equal')
ax.set_xlim(-1.37, 1.89)
ax.set_ylim(-1.36, 1.36)
ax.axis('off')
raios = .105 * np.sqrt(e.to_numpy()/e.max())
# Espaçamento angular ajustado aos raios, em ordem decrescente de área.
ordem_desenho = np.argsort(-e.to_numpy(), kind="stable")
raios_ordenados = raios[ordem_desenho]
intervalos = np.maximum(.078, 2*np.arcsin((raios_ordenados+np.roll(raios_ordenados,-1)+.012)/2))
while intervalos.sum() >= 2*np.pi-.12:
    raios *= .97
    raios_ordenados = raios[ordem_desenho]
    intervalos = np.maximum(.078, 2*np.arcsin((raios_ordenados+np.roll(raios_ordenados,-1)+.012)/2))
raio_maximo_antes_ajuste = float(raios.max())
intervalos += (2*np.pi-intervalos.sum())/67
angulos = np.empty(67)
angulos[ordem_desenho] = np.pi/2 - np.r_[0, np.cumsum(intervalos[:-1])]
posicoes = np.column_stack([np.cos(angulos), np.sin(angulos)])
distancias = np.linalg.norm(posicoes[:, None, :] - posicoes[None, :, :], axis=2)
np.fill_diagonal(distancias, np.inf)
fator_ajuste = min(1., float(np.min(.85*distancias/(raios[:, None]+raios[None, :]))))
raios *= fator_ajuste
normalizacao = Normalize(vmin=0, vmax=float(u.max()))
paleta = plt.get_cmap('viridis_r')
fator_largura = 12.0/max(float(F.to_numpy().max()), float(u.max()))
opacidade = .50
piso_largura_pt = .45
cor_piso = '#bcc3c7'
circulos = [Circle(posicoes[i], raios[i], transform=ax.transData) for i in range(67)]
ligacoes = []

# Camada 1: todas as atribuições, incluindo os laços; menores primeiro.
indices = np.argwhere(F.to_numpy() > 0)
pesos = F.to_numpy()[indices[:, 0], indices[:, 1]]
for k in np.argsort(pesos, kind='stable'):
    i, j = indices[k]
    peso = float(pesos[k])
    largura_calculada = fator_largura*peso
    largura_visual = max(piso_largura_pt, largura_calculada)
    cor_ligacao = cor_piso if largura_calculada < piso_largura_pt else '#52636b'
    if i != j:
        patch = FancyArrowPatch(posicoes[i], posicoes[j],
            connectionstyle='arc3,rad=.085', arrowstyle='-|>',
            patchA=circulos[i], patchB=circulos[j], shrinkA=1, shrinkB=1, mutation_scale=6,
            linewidth=largura_visual, color=cor_ligacao, alpha=opacidade,
            zorder=1)
    else:
        radial = posicoes[i]
        tangente = np.array([-radial[1], radial[0]])
        origem = radial + raios[i]*(np.cos(.7)*radial+np.sin(.7)*tangente)
        destino = radial + raios[i]*(np.cos(.7)*radial-np.sin(.7)*tangente)
        caminho = MplPath([origem,
            radial*(1+raios[i]+.075)+tangente*.048,
            radial*(1+raios[i]+.075)-tangente*.048, destino],
            [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4])
        patch = FancyArrowPatch(path=caminho, arrowstyle='-|>',
            mutation_scale=5, linewidth=largura_visual,
            color=cor_ligacao, alpha=opacidade, zorder=2)
    patch.set_gid(f'F-{codigos[i]}-{codigos[j]}')
    ax.add_patch(patch)
    ligacoes.append((int(i), int(j), peso))

# Camada 2: fechamento das colunas; não é outra parcela somada a F.
saidas = []
for j in range(67):
    destino = (1.47, .17 * posicoes[j, 1])
    patch = FancyArrowPatch(posicoes[j], destino,
        connectionstyle=f'arc3,rad={.10*posicoes[j,1]:.6f}',
        arrowstyle='-|>', patchA=circulos[j], shrinkA=1, shrinkB=2, mutation_scale=7,
        linewidth=fator_largura*float(u.iloc[j]), color='#246f93',
        alpha=opacidade, zorder=2)
    patch.set_gid(f'u-{codigos[j]}-demanda-final')
    ax.add_patch(patch)
    saidas.append(float(u.iloc[j]))

# Camada 3: círculos e identificadores, acima de todas as setas.
for i, codigo in enumerate(codigos):
    circulo = Circle(posicoes[i], raios[i], facecolor=paleta(normalizacao(u.iloc[i])),
                     edgecolor='#27323a', linewidth=.75, zorder=4)
    circulo.set_gid(f'setor-{codigo}')
    ax.add_patch(circulo)
    xy = 1.24 * posicoes[i]
    ax.text(*xy, codigo, ha='center', va='center', fontsize=10.5,
            color='#18252c', zorder=5,
            bbox=dict(facecolor='white', edgecolor='none', pad=.45, alpha=.90))

terminal = FancyBboxPatch((1.47, -.205), .39, .41,
    boxstyle='round,pad=.01,rounding_size=.015', facecolor='#f1f3f5',
    edgecolor='#29363e', linewidth=1.2, zorder=6)
ax.add_patch(terminal)
ax.text(1.665, .065, 'Demanda\nfinal', ha='center', va='center',
        fontsize=17, weight='bold', color='#18252c', zorder=7)
ax.text(1.665, -.10, 'Σu = Σe', ha='center', va='center', fontsize=12,
        color='#43505a', zorder=7)

fig.text(.06, .945, 'Emissões pela produção e pelo consumo', fontsize=29,
         weight='bold', color='#18252c')
fig.text(.06, .912, 'MIP brasileira · 67 setores · 2015 · ordem decrescente de área, no sentido horário', fontsize=20, color='#246f93')
fig.text(.06, .878, 'Área: emissões da produção (e)   |   Cor: atribuição ao consumo (u)   |   Setas: parcelas de F',
         fontsize=14, color='#43505a')

barra_ax = fig.add_axes([.08, .14, .38, .014])
barra = fig.colorbar(plt.cm.ScalarMappable(norm=normalizacao, cmap=paleta),
                     cax=barra_ax, orientation='horizontal')
barra.solids.set_rasterized(False)
barra.ax.xaxis.set_major_formatter(FuncFormatter(lambda n, _: f'{n:,.0f}'.replace(',', '.')))
barra.set_label('Cor dos setores: emissões atribuídas ao consumo (Gg de CO₂)', fontsize=12)

# Legenda de áreas com o mesmo k da figura (conversão dados → pontos).
fig.canvas.draw()
pontos_por_unidade = (ax.transData.transform((1,0))[0]-ax.transData.transform((0,0))[0])*72/fig.dpi
k_area = np.pi*(raio_maximo_antes_ajuste*fator_ajuste*pontos_por_unidade)**2/float(e.max())
legendas = [ax.scatter([], [], s=k_area*v, facecolor='white', edgecolor='#27323a', linewidth=.8)
            for v in [1000, 10000, 50000]]
fig.legend(legendas, ['1.000', '10.000', '50.000'], title='Área: emissões da produção (Gg de CO₂)',
           loc='center', bbox_to_anchor=(.72,.14), ncol=3, frameon=False,
           labelspacing=1.4, handletextpad=1.5, columnspacing=3)
fig.text(.06, .080, 'Cinza: emissor i → demanda final por produtos de j (Fᵢⱼ). Azul: fechamento de cada conta uⱼ no nó terminal.',
         fontsize=12, color='#27323a')
fig.text(.06, .061, 'Cinza claro: ligações com piso visual de 0,45 pt; a espessura dessas linhas não representa sua magnitude.',
         fontsize=12, color='#58646d')
fig.text(.06, .043, 'Atribuições contábeis, não transações diretas ou transporte físico de CO₂. As saídas ao terminal não são somadas novamente a F.',
         fontsize=12, color='#27323a')
fig.text(.06, .028, 'Fonte: MIP/IBGE (2015); coeficientes de Sanguinet e Azzoni (2024), interpolados para 2015. Emissões nacionais estimadas; exportações incluídas.',
         fontsize=10, color='#58646d')

In [4]:
# Auditoria do que efetivamente foi colocado no desenho.
reconstruida = np.zeros((67,67))
for i,j,peso in ligacoes:
    reconstruida[i,j] = peso
np.testing.assert_array_equal(reconstruida, F.to_numpy())
np.testing.assert_array_equal(saidas, u.to_numpy())
np.testing.assert_allclose(np.pi*raios**2/e.to_numpy(),
                           np.repeat(np.pi*raios[0]**2/e.iloc[0],67), rtol=1e-12)
assert np.all(distancias > raios[:,None]+raios[None,:])
assert len(ligacoes) == 4357 and sum(i==j for i,j,_ in ligacoes)==67
assert len(saidas)==67
patches_F = [p for p in ax.patches if (p.get_gid() or '').startswith('F-')]
assert len(patches_F) == 4357
assert all(p.get_linewidth() >= piso_largura_pt for p in patches_F)
assert np.all(np.diff(e.to_numpy()[ordem_desenho]) <= 0)
fig.canvas.draw()
rotulos = [t for t in ax.texts if t.get_text() in codigos]
caixas = [t.get_window_extent(fig.canvas.get_renderer()) for t in rotulos]
assert len(caixas) == 67
assert not any(caixas[i].overlaps(caixas[j]) for i in range(67) for j in range(i+1,67))

stem = pasta/'rede_circular_atribuicao_mip67'
arquivos_figura = {}
for extensao in ['png','svg','pdf']:
    destino = stem.with_suffix('.'+extensao)
    try:
        fig.savefig(destino, dpi=250, facecolor='white')
    except PermissionError:
        # Um visualizador pode bloquear o PDF no Windows; preservar o arquivo aberto.
        if extensao != 'pdf':
            raise
        destino = stem.with_name(stem.name+'_linhas_minimas').with_suffix('.pdf')
        fig.savefig(destino, dpi=250, facecolor='white')
        print('PDF original bloqueado; versao atualizada em:', destino)
    arquivos_figura[extensao] = destino.name
plt.close(fig)
F.to_csv(pasta/'matriz_F_gg_co2.csv', encoding='utf-8-sig', float_format='%.17g')
setores.to_csv(pasta/'setores_atribuicoes.csv', index=False, encoding='utf-8-sig', float_format='%.17g')
metadados = {
    'arquivos_figura':arquivos_figura,
    'n_setores':67, 'parcelas_F_positivas':len(ligacoes), 'autoligacoes':67,
    'saidas_demanda_final':len(saidas), 'total_gg_co2':float(e.sum()),
    'residuo_max_x_Z1_y':float(np.max(np.abs(Z.sum(axis=1)+y-x))),
    'residuo_max_x_Ly':float(np.max(np.abs(L@y-x))),
    'residuo_max_e_F1':float(np.max(np.abs(F.sum(axis=1)-e))),
    'residuo_total_e_u':float(abs(e.sum()-u.sum())),
    'ordem_circulos':'e decrescente, do topo no sentido horario', 'codigos_ordem_desenho':[codigos[i] for i in ordem_desenho],
    'area_proporcional_a':'e', 'cor_linear':'u', 'paleta':'viridis_r',
    'piso_visual_F_pt':piso_largura_pt, 'cor_piso':cor_piso,
    'n_ligacoes_com_piso':int(np.sum(pesos*fator_largura < piso_largura_pt)),
    'fator_largura_pt_por_gg':fator_largura, 'opacidade_setas':opacidade,
    'fator_compatibilidade_monetaria':1, 'interpolacao_phi_peso_2018':4/7,
    'fontes':[dados.entrada(k) for k in ['mip_ibge_2015_67','sanguinet_azzoni_2011','sanguinet_azzoni_2018']],
}
(pasta/'validacao_fontes.json').write_text(json.dumps(metadados,ensure_ascii=False,indent=2),encoding='utf-8')
(pasta/'LEIA-ME.md').write_text(
    '# Rede circular de atribuição — MIP67\n\n'
    + 'Figura atualizada: '+json.dumps(arquivos_figura, ensure_ascii=False)+'\n\n'
    +
    'Reprodução: executar `rede_circular_atribuicao_mip67.ipynb` da raiz do projeto.\n\n'
    'F = diag(phi) L diag(y); e = diag(phi) x = F1; u = Fᵀ1. '
    'Unidade ambiental: Gg de CO₂; x e y: R$ milhões. '
    'Círculos em ordem decrescente de e, do topo no sentido horário. 67 setores sem agregação, 4.357 parcelas positivas de F, incluindo 67 autoligações.\n\n'
    'Área proporcional a e; cor linear em u; largura proporcional à massa, com piso visual de 0,45 pt nas parcelas de F menores que esse limite, em cinza claro. Os pesos numéricos são preservados. '
    'As 67 saídas azuis u são o fechamento visual das colunas de F, não novas emissões. '
    'Não somar os pesos de F com os pesos das saídas. Cada nó reúne os papéis de emissor e destino; '
    'as setas não representam compras diretas, fluxos físicos ou uma cadeia de transformação de F.\n\n'
    'MIP/IBGE de 2015 com exportações incluídas e emissões importadas excluídas. '
    'Coeficientes de Sanguinet e Azzoni (2011/2018) interpolados para 2015, fator monetário 1, '
    'sem deflação adicional: estimativa exploratória, não inventário observado. '
    'Proveniência, hashes e resíduos numéricos em validacao_fontes.json. '
    'A tabela CSV relaciona os códigos visuais aos nomes completos dos setores. '
    'O piso visual evidencia a presença das relações pequenas, sem representar sua magnitude pela espessura.\n',encoding='utf-8')
print(json.dumps({k:v for k,v in metadados.items() if k.startswith('residuo') or k in ['n_setores','parcelas_F_positivas','total_gg_co2']},indent=2))
print('Arquivos:',stem)

PDF original bloqueado; versao atualizada em: C:\GitHub\Redes-1\outputs\rede_circular_atribuicao_mip67\rede_circular_atribuicao_mip67_linhas_minimas.pdf
{
  "n_setores": 67,
  "parcelas_F_positivas": 4357,
  "total_gg_co2": 683696.0414285716,
  "residuo_max_x_Z1_y": 1.1641532182693481e-10,
  "residuo_max_x_Ly": 4.656612873077393e-10,
  "residuo_max_e_F1": 2.9103830456733704e-11,
  "residuo_total_e_u": 1.1641532182693481e-10
}
Arquivos: C:\GitHub\Redes-1\outputs\rede_circular_atribuicao_mip67\rede_circular_atribuicao_mip67


![Rede circular](outputs/rede_circular_atribuicao_mip67/rede_circular_atribuicao_mip67.png)